# Import SLSTR ADF baseline 3.26

This notebook imports the **SLSTR auxiliary data files (ADF) baseline 3.26** into the STAC catalog:

1. it creates (if it does not exist yet) the `copernicus:adf-slstr-baseline-3-26` collection in the catalog with `rs-client` (owner: `copernicus`),
2. it searches all the baseline input files in the ADGS station,
3. it stages all of them into this collection by calling the generic Prefect staging flow (`stage-generic`).

It assumes that all the RS-Python services and the generic staging flow (deployment `stage-generic/Stage Generic`) are deployed and running, and that the API key of the `copernicus` user is stored in the `RSPY_APIKEY` value of the `env-vars-copernicus` Prefect secret block.

## 1. Initialisation

In [ ]:
# Imports
import json
import os
from datetime import datetime, timezone

from prefect.blocks.system import Secret
from pystac import Collection, Extent, ItemCollection, SpatialExtent, TemporalExtent
from requests import HTTPError
from starlette import status

from resources.widget_utils import run_prefect_flow_cmd

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *

In [ ]:
# Catalog owner and collection where the baseline is imported: copernicus:adf-slstr-baseline-3-26
OWNER = "copernicus"
COLLECTION = "adf-slstr-baseline-3-26"

# Name of the generic staging flow deployment, that must be deployed beforehand
STAGING_DEPLOYMENT = "stage-generic/Stage Generic"

# Prefect flow environment arguments: the items are staged in the copernicus catalog.
# The flow uses the copernicus API key read from the env-vars-copernicus Prefect block.
flow_env_args = {
  "env": {
    "owner_id": OWNER,
  },
}

In [ ]:
# Only the copernicus user can create a collection owned by copernicus:
# use RS-Client instances authenticated with the copernicus API key
copernicus_apikey = (await Secret.load(f"env-vars-{OWNER}")).get()["RSPY_APIKEY"]
copernicus_client = RsClient(
    os.environ["RSPY_WEBSITE"],
    rs_server_api_key=copernicus_apikey,
    owner_id=OWNER,
    logger=None,
)
catalog_client = copernicus_client.get_catalog_client()
auxip_client = copernicus_client.get_auxip_client()

## 2. List of the SLSTR baseline 3.26 input files

In [ ]:
BASELINE_FILES = [
    "S3A_SL_1_ADJ_AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_001.SEN3",
    "S3A_SL_1_ANC_AX_20160216T000000_20991231T235959_20190912T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_CDP_AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_001.SEN3",
    "S3A_SL_1_CLO_AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_CLP_AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_001.SEN3",
    "S3A_SL_1_ESSTAX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_001.SEN3",
    "S3A_SL_1_GEC_AX_20190101T000000_20991231T235959_20191010T120000___________________MPC_O_AL_009.SEN3",
    "S3A_SL_1_GEO_AX_20160216T000000_20991231T235959_20190912T120000___________________MPC_O_AL_008.SEN3",
    "S3A_SL_1_IRE_AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_001.SEN3",
    "S3A_SL_1_LCC_AX_20160216T000000_20991231T235959_20201015T120000___________________MPC_O_AL_002.SEN3",
    "S3A_SL_1_NAS4AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_NAS5AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_NAS6AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_NBS4AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_NBS5AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_NBS6AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_010.SEN3",
    "S3A_SL_1_N_F1AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_N_F2AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_N_S1AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_009.SEN3",
    "S3A_SL_1_N_S2AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_009.SEN3",
    "S3A_SL_1_N_S3AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_009.SEN3",
    "S3A_SL_1_N_S7AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_N_S8AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_N_S9AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_OAS4AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_012.SEN3",
    "S3A_SL_1_OAS5AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_012.SEN3",
    "S3A_SL_1_OAS6AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_012.SEN3",
    "S3A_SL_1_OBS4AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_012.SEN3",
    "S3A_SL_1_OBS5AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_012.SEN3",
    "S3A_SL_1_OBS6AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_012.SEN3",
    "S3A_SL_1_O_F1AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_O_F2AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_O_S1AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_011.SEN3",
    "S3A_SL_1_O_S2AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_011.SEN3",
    "S3A_SL_1_O_S3AX_20160418T094050_20991231T235959_20180202T120000___________________MPC_O_AL_011.SEN3",
    "S3A_SL_1_O_S7AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_O_S8AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_O_S9AX_20160216T000000_20991231T235959_20170324T120000___________________MPC_O_AL_006.SEN3",
    "S3A_SL_1_PCP_AX_20160216T000000_20991231T235959_20231222T120000___________________MPC_O_AL_014.SEN3",
    "S3A_SL_1_RTT_AX_20160216T000000_20991231T235959_20180202T120000___________________MPC_O_AL_001.SEN3",
    "S3A_SL_1_VIC_AX_20160216T000000_20991231T235959_20161012T120000___________________MPC_O_AL_004.SEN3",
    "S3__AX___CLM_AX_20000101T000000_20991231T235959_20151214T120000___________________MPC_O_AL_001.SEN3",
    "S3__AX___DEM_AX_20000101T000000_20991231T235959_20151214T120000___________________MPC_O_AL_001.SEN3",
    "S3__AX___LWM_AX_20000101T000000_20991231T235959_20151214T120000___________________MPC_O_AL_001.SEN3",
    "S3__AX___OOM_AX_20000101T000000_20991231T235959_20151214T120000___________________MPC_O_AL_001.SEN3",
    "S3__AX___TRM_AX_20000101T000000_20991231T235959_20151214T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_ADJ_AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_ANC_AX_20180425T000000_20991231T235959_20190912T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_CDP_AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_CLO_AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_CLP_AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_ESSTAX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_GEC_AX_20190101T000000_20991231T235959_20191010T120000___________________MPC_O_AL_004.SEN3",
    "S3B_SL_1_GEO_AX_20180425T000000_20991231T235959_20190912T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_IRE_AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_LCC_AX_20180425T000000_20991231T235959_20201015T120000___________________MPC_O_AL_002.SEN3",
    "S3B_SL_1_NAS4AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_NAS5AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_NAS6AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_NBS4AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_NBS5AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_NBS6AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_N_F1AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_002.SEN3",
    "S3B_SL_1_N_F2AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_002.SEN3",
    "S3B_SL_1_N_S1AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_N_S2AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_N_S3AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_N_S7AX_20180425T000000_20991231T235959_20190912T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_N_S8AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_N_S9AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_OAS4AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_OAS5AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_OAS6AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_OBS4AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_OBS5AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_OBS6AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_O_F1AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_002.SEN3",
    "S3B_SL_1_O_F2AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_O_S1AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_O_S2AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_O_S3AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_003.SEN3",
    "S3B_SL_1_O_S7AX_20180425T000000_20991231T235959_20200410T120000___________________MPC_O_AL_004.SEN3",
    "S3B_SL_1_O_S8AX_20180425T000000_20991231T235959_20180712T120000___________________MPC_O_AL_002.SEN3",
    "S3B_SL_1_O_S9AX_20180425T000000_20991231T235959_20180712T120000___________________MPC_O_AL_002.SEN3",
    "S3B_SL_1_PCP_AX_20180425T000000_20991231T235959_20231222T120000___________________MPC_O_AL_005.SEN3",
    "S3B_SL_1_RTT_AX_20180425T000000_20991231T235959_20180409T120000___________________MPC_O_AL_001.SEN3",
    "S3B_SL_1_VIC_AX_20180425T000000_20991231T235959_20181002T120000___________________MPC_O_AL_002.SEN3",
    "S3__SL_1_DTM_AX_20160216T000000_20991231T235959_20231215T120000___________________MPC_O_AL_001.SEN3",
    "S3__SL_1_PFP_AX_20160216T000000_20991231T235959_20231215T120000___________________MPC_O_AL_001.SEN3",
    "S3__SL_2_LSTBAX_20000101T000000_20991231T235959_20151214T120000___________________MPC_O_AL_001.SEN3",
]

assert len(BASELINE_FILES) == len(set(BASELINE_FILES)), "Duplicated file names"
print(f"{len(BASELINE_FILES)} files in the SLSTR baseline 3.26")

## 3. Create the catalog collection

In [ ]:
print(f"Get or create collection '{OWNER}:{COLLECTION}':")
try:
    collection = Collection(
        id=COLLECTION,
        title="SLSTR ADF baseline 3.26",
        description="Sentinel-3 SLSTR auxiliary data files (ADF) of the processing baseline 3.26, staged from the ADGS.",
        extent=Extent(
            SpatialExtent([[-180.0, -90.0, 180.0, 90.0]]),
            TemporalExtent([[
                datetime(2000, 1, 1, tzinfo=timezone.utc),
                datetime(2099, 12, 31, 23, 59, 59, tzinfo=timezone.utc),
            ]]),
        ),
    )
    response = catalog_client.add_collection(collection, owner_id=OWNER)
    print(response.status_code)

except HTTPError as error:
    # Ignore 409 / conflict error (the collection already existed)
    if error.response.status_code != status.HTTP_409_CONFLICT:
        raise
    print("The collection already exists")

saved_collection = catalog_client.get_collection(COLLECTION, owner_id=OWNER)
print(json.dumps(saved_collection.to_dict(), indent=2))

## 4. Search the baseline files in the ADGS

Each file is searched by its exact name, and we only keep the ADGS items whose id matches.

In [ ]:
adgs_items = {}
for name in BASELINE_FILES:
    results = auxip_client.search(
        method="POST",
        limit=1,
        max_items=1,
        stac_filter={"op": "=", "args": [{"property": "id"}, name]},
    )
    for item in results or []:
        if item.id == name:
            adgs_items[name] = item

missing_in_adgs = [name for name in BASELINE_FILES if name not in adgs_items]
print(f"{len(adgs_items)}/{len(BASELINE_FILES)} files found in the ADGS")
if missing_in_adgs:
    # Don't stop: stage all the files that are available
    print(f"⚠️ {len(missing_in_adgs)} files not found in the ADGS, they will not be staged:")
    print("\n".join(missing_in_adgs))
assert adgs_items, "No file found in the ADGS"

to_stage = ItemCollection(list(adgs_items.values()))

## 5. Stage the baseline files into the catalog collection

In [ ]:
params = {
    **flow_env_args,
    "stac_input": to_stage.to_dict(),
    "catalog_collection_identifier": COLLECTION,
}
# Run the already deployed generic staging flow and wait for its completion
state = await run_prefect_flow_cmd(
    deploy_name=STAGING_DEPLOYMENT,
    params=params,
    flow_run_name="stage-adf-slstr-baseline-3-26",
)
if state is not None:
    flow_run_id = state.state_details.flow_run_id
    print(f"Flow run id: {flow_run_id!r}")
    if state.is_failed() or state.is_crashed():
        # Retrieve logs of failed flow run
        response = http_session.get(f"{os.environ['PREFECT_API_URL']}/flow_runs/{flow_run_id}/logs/download")
        response.raise_for_status()
        raise RuntimeError(f"Prefect flow {flow_run_id} failed.\n\nLogs:\n{response.text}")

## 6. Check the staged items in the catalog

In [ ]:
staged = catalog_client.search(
    owner_id=OWNER,
    collections=[COLLECTION],
    limit=100,
).items
staged_names = {item.id for item in staged}
print(f"{len(staged_names)} items in the '{OWNER}:{COLLECTION}' collection")

# All the files found in the ADGS must have been staged
not_staged = sorted(set(adgs_items) - staged_names)
assert not not_staged, "Files found in the ADGS but not staged in the catalog:\n" + "\n".join(not_staged)
print(f"✅ All the {len(adgs_items)} files found in the ADGS are staged in the catalog")

# Reminder of the baseline files that are still missing
missing = sorted(set(BASELINE_FILES) - staged_names)
if missing:
    print(f"⚠️ {len(missing)}/{len(BASELINE_FILES)} SLSTR baseline 3.26 files are still missing in the catalog:")
    print("\n".join(missing))